# 🕵️ Dia 56 — Normalizando o tempo e correlacionando logs entre hosts

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 56* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-56.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sua planilha pôs o atacante no webmail às 9h32 e o clique da caixa às 11h19. Rafa: *"O golpista entrou antes de roubar a senha?"* Não: o *firewall* grava em Brasília sem dizer. Hoje você prova cada relógio.

Neste notebook você vai:

1. **Achar todos os pares *firewall* × `sshd`** pela porta de origem e medir a diferença de cada um.
2. **Confirmar UTC-4 com `zoneinfo`** e medir os 83 s do `fs01`.
3. **Rodar o `merge_timeline.py`** do original e conferir as 297 linhas.
4. **Ver a linha do tempo bruta ao lado da corrigida.**
5. **Gerar o grafo** com a aresta inferida tracejada.
6. **Repetir com os três relógios da Serra Azul.**
7. **Jogar o quiz do relógio.**


In [ ]:
# ------------------------------------------------------------
# RECEBER O EVID-001 (QUATRO LOGS DO LAB-P4)
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-4-digital-forensics-incident-investigation/resources/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['case-lab-p4/logs/bastion01-auth.log'] = ''.join((
    'Mar 13 14:02:11 bastion01 sshd[21807]: Accepted publickey for ops.admin from 10.10.50.12 port 51544 ssh2: ED25519 SHA256:q8m1Xx0lab0synthetic0key0fingerprint0AAAA\n'
    'Mar 13 14:02:12 bastion01 sshd[21807]: pam_unix(sshd:session): session opened for user ops.admin(uid=1001) by (uid=0)\n'
    'Mar 13 16:47:39 bastion01 sshd[21814]: Accepted publickey for ops.admin from 10.10.50.12 port 52010 ssh2: ED25519 SHA256:q8m1Xx0lab0synthetic0key0fingerprint0AAAA\n'
    'Mar 13 16:47:40 bastion01 sshd[21814]: pam_unix(sshd:session): session opened for user ops.admin(uid=1001) by (uid=0)\n'
    'Mar 14 02:51:03 bastion01 sshd[21815]: Failed password for invalid user admin from 203.0.113.45 port 49208 ssh2\n'
    'Mar 14 02:51:15 bastion01 sshd[21816]: Failed password for root from 203.0.113.45 port 49212 ssh2\n'
    'Mar 14 02:51:24 bastion01 sshd[21817]: Failed password for invalid user backup from 203.0.113.45 port 49216 ssh2\n'
    'Mar 14 02:51:34 bastion01 sshd[21818]: Failed password for invalid user oracle from 203.0.113.45 port 49224 ssh2\n'
    'Mar 14 02:51:49 bastion01 sshd[21819]: Failed password for svc_backup from 203.0.113.45 port 49230 ssh2\n'
    'Mar 14 02:52:04 bastion01 sshd[21820]: Failed password for invalid user test from 203.0.113.45 port 49237 ssh2\n'
    'Mar 14 02:52:18 bastion01 sshd[21821]: Failed password for invalid user ubuntu from 203.0.113.45 port 49245 ssh2\n'
    'Mar 14 02:52:30 bastion01 sshd[21822]: Failed password for invalid user ftpuser from 203.0.113.45 port 49249 ssh2\n'
    'Mar 14 02:52:39 bastion01 sshd[21823]: Failed password for invalid user admin from 203.0.113.45 port 49252 ssh2\n'
    'Mar 14 02:52:52 bastion01 sshd[21824]: Failed password for root from 203.0.113.45 port 49256 ssh2\n'
    'Mar 14 02:53:02 bastion01 sshd[21825]: Failed password for invalid user backup from 203.0.113.45 port 49264 ssh2\n'
    'Mar 14 02:53:16 bastion01 sshd[21826]: Failed password for invalid user oracle from 203.0.113.45 port 49268 ssh2\n'
    'Mar 14 02:53:29 bastion01 sshd[21827]: Failed password for svc_backup from 203.0.113.45 port 49269 ssh2\n'
    'Mar 14 02:53:42 bastion01 sshd[21828]: Failed password for invalid user test from 203.0.113.45 port 49278 ssh2\n'
    'Mar 14 02:53:51 bastion01 sshd[21829]: Failed password for invalid user ubuntu from 203.0.113.45 port 49279 ssh2\n'
    'Mar 14 02:54:03 bastion01 sshd[21830]: Failed password for invalid user ftpuser from 203.0.113.45 port 49287 ssh2\n'
    'Mar 14 02:54:18 bastion01 sshd[21831]: Failed password for invalid user admin from 203.0.113.45 port 49296 ssh2\n'
    'Mar 14 02:54:32 bastion01 sshd[21832]: Failed password for root from 203.0.113.45 port 49303 ssh2\n'
    'Mar 14 02:54:43 bastion01 sshd[21833]: Failed password for invalid user backup from 203.0.113.45 port 49311 ssh2\n'
    'Mar 14 02:54:53 bastion01 sshd[21834]: Failed password for invalid user oracle from 203.0.113.45 port 49316 ssh2\n'
    'Mar 14 02:55:04 bastion01 sshd[21835]: Failed password for svc_backup from 203.0.113.45 port 49321 ssh2\n'
    'Mar 14 02:55:16 bastion01 sshd[21836]: Failed password for invalid user test from 203.0.113.45 port 49330 ssh2\n'
    'Mar 14 02:55:25 bastion01 sshd[21837]: Failed password for invalid user ubuntu from 203.0.113.45 port 49337 ssh2\n'
    'Mar 14 02:55:37 bastion01 sshd[21838]: Failed password for invalid user ftpuser from 203.0.113.45 port 49343 ssh2\n'
    'Mar 14 02:55:50 bastion01 sshd[21839]: Failed password for invalid user admin from 203.0.113.45 port 49347 ssh2\n'
    'Mar 14 02:56:02 bastion01 sshd[21840]: Failed password for root from 203.0.113.45 port 49348 ssh2\n'
    'Mar 14 02:56:14 bastion01 sshd[21841]: Failed password for invalid user backup from 203.0.113.45 port 49357 ssh2\n'
    'Mar 14 02:56:27 bastion01 sshd[21842]: Failed password for invalid user oracle from 203.0.113.45 port 49359 ssh2\n'
    'Mar 14 02:56:42 bastion01 sshd[21843]: Failed password for svc_backup from 203.0.113.45 port 49368 ssh2\n'
    'Mar 14 02:56:54 bastion01 sshd[21844]: Failed password for invalid user test from 203.0.113.45 port 49374 ssh2\n'
    'Mar 14 02:57:08 bastion01 sshd[21845]: Failed password for invalid user ubuntu from 203.0.113.45 port 49376 ssh2\n'
    'Mar 14 02:57:22 bastion01 sshd[21846]: Failed password for invalid user ftpuser from 203.0.113.45 port 49379 ssh2\n'
    'Mar 14 02:57:34 bastion01 sshd[21847]: Failed password for invalid user admin from 203.0.113.45 port 49383 ssh2\n'
    'Mar 14 02:57:45 bastion01 sshd[21848]: Failed password for root from 203.0.113.45 port 49387 ssh2\n'
    'Mar 14 02:57:59 bastion01 sshd[21849]: Failed password for invalid user backup from 203.0.113.45 port 49395 ssh2\n'
    'Mar 14 02:58:11 bastion01 sshd[21850]: Failed password for invalid user oracle from 203.0.113.45 port 49398 ssh2\n'
    'Mar 14 02:58:26 bastion01 sshd[21851]: Failed password for svc_backup from 203.0.113.45 port 49403 ssh2\n'
    'Mar 14 02:58:38 bastion01 sshd[21852]: Failed password for invalid user test from 203.0.113.45 port 49410 ssh2\n'
    'Mar 14 02:58:49 bastion01 sshd[21853]: Failed password for invalid user ubuntu from 203.0.113.45 port 49411 ssh2\n'
    'Mar 14 02:59:03 bastion01 sshd[21854]: Failed password for invalid user ftpuser from 203.0.113.45 port 49419 ssh2\n'
    'Mar 14 02:59:14 bastion01 sshd[21855]: Failed password for invalid user admin from 203.0.113.45 port 49422 ssh2\n'
    'Mar 14 02:59:28 bastion01 sshd[21856]: Failed password for root from 203.0.113.45 port 49430 ssh2\n'
    'Mar 14 02:59:43 bastion01 sshd[21857]: Failed password for invalid user backup from 203.0.113.45 port 49434 ssh2\n'
    'Mar 14 02:59:54 bastion01 sshd[21858]: Failed password for invalid user oracle from 203.0.113.45 port 49436 ssh2\n'
    'Mar 14 03:00:09 bastion01 sshd[21859]: Failed password for svc_backup from 203.0.113.45 port 49440 ssh2\n'
    'Mar 14 03:00:24 bastion01 sshd[21860]: Failed password for invalid user test from 203.0.113.45 port 49443 ssh2\n'
    'Mar 14 03:00:39 bastion01 sshd[21861]: Failed password for invalid user ubuntu from 203.0.113.45 port 49447 ssh2\n'
    'Mar 14 03:00:48 bastion01 sshd[21862]: Failed password for invalid user ftpuser from 203.0.113.45 port 49450 ssh2\n'
    'Mar 14 03:01:02 bastion01 sshd[21863]: Failed password for invalid user admin from 203.0.113.45 port 49458 ssh2\n'
    'Mar 14 03:01:17 bastion01 sshd[21864]: Failed password for root from 203.0.113.45 port 49461 ssh2\n'
    'Mar 14 03:01:29 bastion01 sshd[21865]: Failed password for invalid user backup from 203.0.113.45 port 49470 ssh2\n'
    'Mar 14 03:01:39 bastion01 sshd[21866]: Failed password for invalid user oracle from 203.0.113.45 port 49478 ssh2\n'
    'Mar 14 03:01:53 bastion01 sshd[21867]: Failed password for svc_backup from 203.0.113.45 port 49486 ssh2\n'
    'Mar 14 03:02:08 bastion01 sshd[21868]: Failed password for invalid user test from 203.0.113.45 port 49491 ssh2\n'
    'Mar 14 03:02:23 bastion01 sshd[21869]: Failed password for invalid user ubuntu from 203.0.113.45 port 49499 ssh2\n'
    'Mar 14 03:02:36 bastion01 sshd[21870]: Failed password for invalid user ftpuser from 203.0.113.45 port 49504 ssh2\n'
    'Mar 14 03:02:51 bastion01 sshd[21871]: Failed password for invalid user admin from 203.0.113.45 port 49511 ssh2\n'
    'Mar 14 03:03:03 bastion01 sshd[21872]: Failed password for root from 203.0.113.45 port 49512 ssh2\n'
    'Mar 14 03:03:13 bastion01 sshd[21873]: Failed password for invalid user backup from 203.0.113.45 port 49517 ssh2\n'
    'Mar 14 03:03:28 bastion01 sshd[21874]: Failed password for invalid user oracle from 203.0.113.45 port 49524 ssh2\n'
    'Mar 14 03:03:40 bastion01 sshd[21875]: Failed password for svc_backup from 203.0.113.45 port 49525 ssh2\n'
    'Mar 14 03:03:55 bastion01 sshd[21876]: Failed password for invalid user test from 203.0.113.45 port 49534 ssh2\n'
    'Mar 14 03:04:05 bastion01 sshd[21877]: Failed password for invalid user ubuntu from 203.0.113.45 port 49539 ssh2\n'
    'Mar 14 03:04:17 bastion01 sshd[21878]: Failed password for invalid user ftpuser from 203.0.113.45 port 49547 ssh2\n'
    'Mar 14 03:04:31 bastion01 sshd[21879]: Failed password for invalid user admin from 203.0.113.45 port 49556 ssh2\n'
    'Mar 14 03:04:46 bastion01 sshd[21880]: Failed password for root from 203.0.113.45 port 49561 ssh2\n'
    'Mar 14 03:05:00 bastion01 sshd[21881]: Failed password for invalid user backup from 203.0.113.45 port 49565 ssh2\n'
    'Mar 14 03:05:14 bastion01 sshd[21882]: Failed password for invalid user oracle from 203.0.113.45 port 49568 ssh2\n'
    'Mar 14 03:05:23 bastion01 sshd[21883]: Failed password for svc_backup from 203.0.113.45 port 49576 ssh2\n'
    'Mar 14 03:05:32 bastion01 sshd[21884]: Failed password for invalid user test from 203.0.113.45 port 49577 ssh2\n'
    'Mar 14 03:05:44 bastion01 sshd[21885]: Failed password for invalid user ubuntu from 203.0.113.45 port 49582 ssh2\n'
    'Mar 14 03:05:55 bastion01 sshd[21886]: Failed password for invalid user ftpuser from 203.0.113.45 port 49589 ssh2\n'
    'Mar 14 03:06:06 bastion01 sshd[21887]: Failed password for invalid user admin from 203.0.113.45 port 49593 ssh2\n'
    'Mar 14 03:06:21 bastion01 sshd[21888]: Failed password for root from 203.0.113.45 port 49598 ssh2\n'
    'Mar 14 03:06:34 bastion01 sshd[21889]: Failed password for invalid user backup from 203.0.113.45 port 49602 ssh2\n'
    'Mar 14 03:06:48 bastion01 sshd[21890]: Failed password for invalid user oracle from 203.0.113.45 port 49605 ssh2\n'
    'Mar 14 03:06:57 bastion01 sshd[21891]: Failed password for svc_backup from 203.0.113.45 port 49614 ssh2\n'
    'Mar 14 03:07:07 bastion01 sshd[21892]: Failed password for invalid user test from 203.0.113.45 port 49617 ssh2\n'
    'Mar 14 03:07:18 bastion01 sshd[21893]: Failed password for invalid user ubuntu from 203.0.113.45 port 49625 ssh2\n'
    'Mar 14 03:07:30 bastion01 sshd[21894]: Failed password for invalid user ftpuser from 203.0.113.45 port 49633 ssh2\n'
    'Mar 14 03:07:39 bastion01 sshd[21895]: Failed password for invalid user admin from 203.0.113.45 port 49636 ssh2\n'
    'Mar 14 03:07:51 bastion01 sshd[21896]: Failed password for root from 203.0.113.45 port 49640 ssh2\n'
    'Mar 14 03:08:01 bastion01 sshd[21897]: Failed password for invalid user backup from 203.0.113.45 port 49647 ssh2\n'
    'Mar 14 03:08:10 bastion01 sshd[21898]: Failed password for invalid user oracle from 203.0.113.45 port 49649 ssh2\n'
    'Mar 14 03:08:19 bastion01 sshd[21899]: Failed password for svc_backup from 203.0.113.45 port 49651 ssh2\n'
    'Mar 14 03:08:28 bastion01 sshd[21900]: Failed password for invalid user test from 203.0.113.45 port 49655 ssh2\n'
    'Mar 14 03:08:42 bastion01 sshd[21901]: Failed password for invalid user ubuntu from 203.0.113.45 port 49657 ssh2\n'
    'Mar 14 03:08:54 bastion01 sshd[21902]: Failed password for invalid user ftpuser from 203.0.113.45 port 49658 ssh2\n'
    'Mar 14 03:09:03 bastion01 sshd[21903]: Failed password for invalid user admin from 203.0.113.45 port 49659 ssh2\n'
    'Mar 14 03:09:14 bastion01 sshd[21904]: Failed password for root from 203.0.113.45 port 49664 ssh2\n'
    'Mar 14 03:09:25 bastion01 sshd[21905]: Failed password for invalid user backup from 203.0.113.45 port 49665 ssh2\n'
    'Mar 14 03:09:37 bastion01 sshd[21906]: Failed password for invalid user oracle from 203.0.113.45 port 49673 ssh2\n'
    'Mar 14 03:09:51 bastion01 sshd[21907]: Failed password for svc_backup from 203.0.113.45 port 49681 ssh2\n'
    'Mar 14 03:10:00 bastion01 sshd[21908]: Failed password for invalid user test from 203.0.113.45 port 49689 ssh2\n'
    'Mar 14 03:10:10 bastion01 sshd[21909]: Failed password for invalid user ubuntu from 203.0.113.45 port 49696 ssh2\n'
    'Mar 14 03:10:19 bastion01 sshd[21910]: Failed password for invalid user ftpuser from 203.0.113.45 port 49703 ssh2\n'
    'Mar 14 03:10:33 bastion01 sshd[21911]: Failed password for invalid user admin from 203.0.113.45 port 49712 ssh2\n'
    'Mar 14 03:10:47 bastion01 sshd[21912]: Failed password for root from 203.0.113.45 port 49713 ssh2\n'
    'Mar 14 03:11:02 bastion01 sshd[21913]: Failed password for invalid user backup from 203.0.113.45 port 49720 ssh2\n'
    'Mar 14 03:11:12 bastion01 sshd[21914]: Failed password for invalid user oracle from 203.0.113.45 port 49727 ssh2\n'
    'Mar 14 03:11:22 bastion01 sshd[21915]: Failed password for svc_backup from 203.0.113.45 port 49734 ssh2\n'
    'Mar 14 03:11:36 bastion01 sshd[21916]: Failed password for invalid user test from 203.0.113.45 port 49742 ssh2\n'
    'Mar 14 03:11:49 bastion01 sshd[21917]: Failed password for invalid user ubuntu from 203.0.113.45 port 49744 ssh2\n'
    'Mar 14 03:12:04 bastion01 sshd[21918]: Failed password for invalid user ftpuser from 203.0.113.45 port 49745 ssh2\n'
    'Mar 14 03:12:13 bastion01 sshd[21919]: Failed password for invalid user admin from 203.0.113.45 port 49751 ssh2\n'
    'Mar 14 03:12:23 bastion01 sshd[21920]: Failed password for root from 203.0.113.45 port 49753 ssh2\n'
    'Mar 14 03:12:37 bastion01 sshd[21921]: Failed password for invalid user backup from 203.0.113.45 port 49755 ssh2\n'
    'Mar 14 03:12:47 bastion01 sshd[21922]: Failed password for invalid user oracle from 203.0.113.45 port 49759 ssh2\n'
    'Mar 14 03:13:01 bastion01 sshd[21923]: Failed password for svc_backup from 203.0.113.45 port 49760 ssh2\n'
    'Mar 14 03:13:15 bastion01 sshd[21924]: Failed password for invalid user test from 203.0.113.45 port 49766 ssh2\n'
    'Mar 14 03:13:24 bastion01 sshd[21925]: Failed password for invalid user ubuntu from 203.0.113.45 port 49774 ssh2\n'
    'Mar 14 03:13:36 bastion01 sshd[21926]: Failed password for invalid user ftpuser from 203.0.113.45 port 49783 ssh2\n'
    'Mar 14 03:14:07 bastion01 sshd[21927]: Accepted password for svc_backup from 203.0.113.45 port 50122 ssh2\n'
    'Mar 14 03:14:08 bastion01 sshd[21927]: pam_unix(sshd:session): session opened for user svc_backup(uid=1107) by (uid=0)\n'
    'Mar 14 03:14:08 bastion01 systemd-logind[642]: New session 318 of user svc_backup.\n'
    'Mar 14 03:52:44 bastion01 sshd[21927]: pam_unix(sshd:session): session closed for user svc_backup\n'
))
RESERVA['case-lab-p4/logs/fs01-auth.log'] = ''.join((
    'Mar 13 09:16:25 fs01 sshd[1188]: Accepted publickey for ops.admin from 10.10.50.12 port 50912 ssh2: ED25519 SHA256:q8m1Xx0lab0synthetic0key0fingerprint0AAAA\n'
    'Mar 14 03:20:45 fs01 sshd[30411]: Accepted publickey for svc_backup from 10.10.20.5 port 41766 ssh2: RSA SHA256:Zb7synthetic0lab0svcbackup0key0BBBBBBBBBB\n'
    'Mar 14 03:20:46 fs01 sshd[30411]: pam_unix(sshd:session): session opened for user svc_backup(uid=1107) by (uid=0)\n'
    'Mar 14 03:33:03 fs01 sudo[30502]: svc_backup : TTY=pts/1 ; PWD=/home/svc_backup ; USER=root ; COMMAND=/usr/bin/tar -czf /var/tmp/.q1.tgz /srv/docportal/finance/2026-Q1\n'
    'Mar 14 03:33:03 fs01 sudo[30502]: pam_unix(sudo:session): session opened for user root(uid=0) by svc_backup(uid=1107)\n'
    'Mar 14 03:34:28 fs01 sudo[30502]: pam_unix(sudo:session): session closed for user root\n'
    'Mar 14 03:51:21 fs01 sshd[30411]: pam_unix(sshd:session): session closed for user svc_backup\n'
))
RESERVA['case-lab-p4/logs/docportal-app.jsonl'] = ''.join((
    '{"ts": "2026-03-13T13:22:23Z", "level": "INFO", "event": "document.view", "user": "acct.clerk01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1678.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T13:53:23Z", "level": "INFO", "event": "document.view", "user": "acct.lead02", "src_ip": "10.10.40.57", "path": "/finance/2026-Q1/inv-1836.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T14:26:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1493.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T14:27:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1100.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T14:46:23Z", "level": "INFO", "event": "document.view", "user": "acct.lead02", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1304.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T14:46:23Z", "level": "INFO", "event": "document.view", "user": "acct.clerk01", "src_ip": "10.10.40.57", "path": "/finance/2026-Q1/inv-1683.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T15:02:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1153.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T15:36:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1823.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:11:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1419.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:16:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1024.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:30:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1481.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:39:23Z", "level": "INFO", "event": "document.view", "user": "acct.lead02", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1686.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:47:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1461.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T17:00:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1438.pdf", "count": 1}\n'
    '{"ts": "2026-03-14T03:24:33Z", "level": "INFO", "event": "auth.login", "user": "svc_backup", "src_ip": "127.0.0.1", "method": "local-token", "count": 1}\n'
    '{"ts": "2026-03-14T03:25:11Z", "level": "WARN", "event": "document.bulk_export", "user": "svc_backup", "src_ip": "127.0.0.1", "path": "/finance/2026-Q1", "count": 412, "bytes": 48006112}\n'
    '{"ts": "2026-03-14T03:25:25Z", "level": "INFO", "event": "export.complete", "user": "svc_backup", "path": "/srv/docportal/finance/2026-Q1", "count": 412}\n'
))
RESERVA['case-lab-p4/logs/firewall.log'] = ''.join((
    '2026-03-13 16:07:00 fw01 ACCEPT TCP 10.10.40.57:49982 -> 192.0.2.20:443 rule=allow-https-out bytes=45649\n'
    '2026-03-13 16:19:00 fw01 ACCEPT TCP 10.10.50.12:50042 -> 192.0.2.10:443 rule=allow-https-out bytes=11956\n'
    '2026-03-13 16:40:00 fw01 ACCEPT TCP 10.10.50.12:59993 -> 192.0.2.30:443 rule=allow-https-out bytes=11441\n'
    '2026-03-13 16:54:00 fw01 ACCEPT TCP 10.10.40.57:49616 -> 192.0.2.36:443 rule=allow-https-out bytes=78463\n'
    '2026-03-13 16:58:00 fw01 ACCEPT TCP 10.10.40.61:55186 -> 192.0.2.26:443 rule=allow-https-out bytes=3348\n'
    '2026-03-13 17:23:00 fw01 ACCEPT TCP 10.10.40.61:43475 -> 192.0.2.18:443 rule=allow-https-out bytes=28997\n'
    '2026-03-13 17:27:00 fw01 ACCEPT TCP 10.10.50.12:58821 -> 192.0.2.31:443 rule=allow-https-out bytes=20745\n'
    '2026-03-13 17:36:00 fw01 ACCEPT TCP 10.10.40.61:44782 -> 192.0.2.30:443 rule=allow-https-out bytes=74055\n'
    '2026-03-13 17:42:00 fw01 ACCEPT TCP 10.10.50.12:49971 -> 192.0.2.40:443 rule=allow-https-out bytes=2724\n'
    '2026-03-13 18:12:00 fw01 ACCEPT TCP 10.10.50.12:46834 -> 192.0.2.20:443 rule=allow-https-out bytes=37289\n'
    '2026-03-13 18:14:00 fw01 ACCEPT TCP 10.10.40.61:58905 -> 192.0.2.24:443 rule=allow-https-out bytes=24639\n'
    '2026-03-13 18:57:00 fw01 ACCEPT TCP 10.10.50.12:41396 -> 192.0.2.16:443 rule=allow-https-out bytes=24890\n'
    '2026-03-13 19:37:00 fw01 ACCEPT TCP 10.10.40.61:57120 -> 192.0.2.31:443 rule=allow-https-out bytes=81741\n'
    '2026-03-13 19:43:00 fw01 ACCEPT TCP 10.10.50.12:57897 -> 192.0.2.27:443 rule=allow-https-out bytes=3149\n'
    '2026-03-13 20:17:00 fw01 ACCEPT TCP 10.10.40.57:57888 -> 192.0.2.39:443 rule=allow-https-out bytes=28850\n'
    '2026-03-13 20:26:00 fw01 ACCEPT TCP 10.10.40.61:47688 -> 192.0.2.16:443 rule=allow-https-out bytes=45144\n'
    '2026-03-13 20:42:00 fw01 ACCEPT TCP 10.10.50.12:42196 -> 192.0.2.17:443 rule=allow-https-out bytes=70580\n'
    '2026-03-13 21:00:00 fw01 DROP TCP 198.51.100.71:56074 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 21:02:00 fw01 DROP TCP 198.51.100.68:51554 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 21:04:00 fw01 ACCEPT TCP 10.10.40.61:59948 -> 192.0.2.13:443 rule=allow-https-out bytes=41375\n'
    '2026-03-13 21:06:00 fw01 ACCEPT TCP 10.10.50.12:50162 -> 192.0.2.32:443 rule=allow-https-out bytes=37853\n'
    '2026-03-13 21:12:00 fw01 ACCEPT TCP 10.10.40.57:46775 -> 192.0.2.34:443 rule=allow-https-out bytes=20418\n'
    '2026-03-13 21:17:00 fw01 DROP TCP 198.51.100.86:49921 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 21:27:00 fw01 DROP TCP 198.51.100.81:52296 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 21:39:00 fw01 ACCEPT TCP 10.10.50.12:40560 -> 192.0.2.29:443 rule=allow-https-out bytes=44941\n'
    '2026-03-13 21:41:00 fw01 ACCEPT TCP 10.10.50.12:51843 -> 192.0.2.31:443 rule=allow-https-out bytes=64882\n'
    '2026-03-13 21:45:00 fw01 ACCEPT TCP 10.10.40.61:45200 -> 192.0.2.34:443 rule=allow-https-out bytes=5749\n'
    '2026-03-13 21:51:00 fw01 ACCEPT TCP 10.10.50.12:49874 -> 192.0.2.13:443 rule=allow-https-out bytes=23423\n'
    '2026-03-13 21:56:00 fw01 ACCEPT TCP 10.10.50.12:48605 -> 192.0.2.35:443 rule=allow-https-out bytes=18536\n'
    '2026-03-13 22:14:00 fw01 ACCEPT TCP 10.10.40.57:50184 -> 192.0.2.35:443 rule=allow-https-out bytes=65725\n'
    '2026-03-13 22:44:00 fw01 ACCEPT TCP 10.10.40.57:58676 -> 192.0.2.19:443 rule=allow-https-out bytes=21220\n'
    '2026-03-13 22:49:00 fw01 ACCEPT TCP 10.10.40.57:50528 -> 192.0.2.25:443 rule=allow-https-out bytes=23310\n'
    '2026-03-13 22:51:02 fw01 ACCEPT TCP 203.0.113.45:49208 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:51:14 fw01 ACCEPT TCP 203.0.113.45:49212 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:51:23 fw01 ACCEPT TCP 203.0.113.45:49216 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:51:33 fw01 ACCEPT TCP 203.0.113.45:49224 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:51:48 fw01 ACCEPT TCP 203.0.113.45:49230 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:03 fw01 ACCEPT TCP 203.0.113.45:49237 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:17 fw01 ACCEPT TCP 203.0.113.45:49245 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:29 fw01 ACCEPT TCP 203.0.113.45:49249 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:38 fw01 ACCEPT TCP 203.0.113.45:49252 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:51 fw01 ACCEPT TCP 203.0.113.45:49256 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:01 fw01 ACCEPT TCP 203.0.113.45:49264 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:15 fw01 ACCEPT TCP 203.0.113.45:49268 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:28 fw01 ACCEPT TCP 203.0.113.45:49269 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:41 fw01 ACCEPT TCP 203.0.113.45:49278 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:50 fw01 ACCEPT TCP 203.0.113.45:49279 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:02 fw01 ACCEPT TCP 203.0.113.45:49287 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:17 fw01 ACCEPT TCP 203.0.113.45:49296 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:31 fw01 ACCEPT TCP 203.0.113.45:49303 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:42 fw01 ACCEPT TCP 203.0.113.45:49311 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:52 fw01 ACCEPT TCP 203.0.113.45:49316 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:03 fw01 ACCEPT TCP 203.0.113.45:49321 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:15 fw01 ACCEPT TCP 203.0.113.45:49330 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:24 fw01 ACCEPT TCP 203.0.113.45:49337 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:36 fw01 ACCEPT TCP 203.0.113.45:49343 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:49 fw01 ACCEPT TCP 203.0.113.45:49347 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:01 fw01 ACCEPT TCP 203.0.113.45:49348 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:13 fw01 ACCEPT TCP 203.0.113.45:49357 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:26 fw01 ACCEPT TCP 203.0.113.45:49359 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:41 fw01 ACCEPT TCP 203.0.113.45:49368 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:53 fw01 ACCEPT TCP 203.0.113.45:49374 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:00 fw01 ACCEPT TCP 10.10.40.61:47756 -> 192.0.2.29:443 rule=allow-https-out bytes=85848\n'
    '2026-03-13 22:57:07 fw01 ACCEPT TCP 203.0.113.45:49376 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:21 fw01 ACCEPT TCP 203.0.113.45:49379 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:33 fw01 ACCEPT TCP 203.0.113.45:49383 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:44 fw01 ACCEPT TCP 203.0.113.45:49387 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:58 fw01 ACCEPT TCP 203.0.113.45:49395 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:58:10 fw01 ACCEPT TCP 203.0.113.45:49398 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:58:25 fw01 ACCEPT TCP 203.0.113.45:49403 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:58:37 fw01 ACCEPT TCP 203.0.113.45:49410 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:58:48 fw01 ACCEPT TCP 203.0.113.45:49411 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:02 fw01 ACCEPT TCP 203.0.113.45:49419 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:13 fw01 ACCEPT TCP 203.0.113.45:49422 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:27 fw01 ACCEPT TCP 203.0.113.45:49430 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:42 fw01 ACCEPT TCP 203.0.113.45:49434 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:53 fw01 ACCEPT TCP 203.0.113.45:49436 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:00:08 fw01 ACCEPT TCP 203.0.113.45:49440 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:00:23 fw01 ACCEPT TCP 203.0.113.45:49443 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:00:38 fw01 ACCEPT TCP 203.0.113.45:49447 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:00:47 fw01 ACCEPT TCP 203.0.113.45:49450 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:01 fw01 ACCEPT TCP 203.0.113.45:49458 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:16 fw01 ACCEPT TCP 203.0.113.45:49461 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:28 fw01 ACCEPT TCP 203.0.113.45:49470 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:38 fw01 ACCEPT TCP 203.0.113.45:49478 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:52 fw01 ACCEPT TCP 203.0.113.45:49486 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:02:07 fw01 ACCEPT TCP 203.0.113.45:49491 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:02:22 fw01 ACCEPT TCP 203.0.113.45:49499 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:02:35 fw01 ACCEPT TCP 203.0.113.45:49504 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:02:50 fw01 ACCEPT TCP 203.0.113.45:49511 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:02 fw01 ACCEPT TCP 203.0.113.45:49512 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:12 fw01 ACCEPT TCP 203.0.113.45:49517 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:27 fw01 ACCEPT TCP 203.0.113.45:49524 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:39 fw01 ACCEPT TCP 203.0.113.45:49525 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:54 fw01 ACCEPT TCP 203.0.113.45:49534 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:04 fw01 ACCEPT TCP 203.0.113.45:49539 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:16 fw01 ACCEPT TCP 203.0.113.45:49547 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:30 fw01 ACCEPT TCP 203.0.113.45:49556 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:45 fw01 ACCEPT TCP 203.0.113.45:49561 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:59 fw01 ACCEPT TCP 203.0.113.45:49565 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:13 fw01 ACCEPT TCP 203.0.113.45:49568 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:22 fw01 ACCEPT TCP 203.0.113.45:49576 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:31 fw01 ACCEPT TCP 203.0.113.45:49577 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:43 fw01 ACCEPT TCP 203.0.113.45:49582 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:54 fw01 ACCEPT TCP 203.0.113.45:49589 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:05 fw01 ACCEPT TCP 203.0.113.45:49593 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:20 fw01 ACCEPT TCP 203.0.113.45:49598 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:33 fw01 ACCEPT TCP 203.0.113.45:49602 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:47 fw01 ACCEPT TCP 203.0.113.45:49605 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:56 fw01 ACCEPT TCP 203.0.113.45:49614 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:06 fw01 ACCEPT TCP 203.0.113.45:49617 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:17 fw01 ACCEPT TCP 203.0.113.45:49625 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:29 fw01 ACCEPT TCP 203.0.113.45:49633 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:38 fw01 ACCEPT TCP 203.0.113.45:49636 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:50 fw01 ACCEPT TCP 203.0.113.45:49640 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:00 fw01 ACCEPT TCP 203.0.113.45:49647 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:09 fw01 ACCEPT TCP 203.0.113.45:49649 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:18 fw01 ACCEPT TCP 203.0.113.45:49651 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:27 fw01 ACCEPT TCP 203.0.113.45:49655 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:41 fw01 ACCEPT TCP 203.0.113.45:49657 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:53 fw01 ACCEPT TCP 203.0.113.45:49658 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:02 fw01 ACCEPT TCP 203.0.113.45:49659 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:13 fw01 ACCEPT TCP 203.0.113.45:49664 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:24 fw01 ACCEPT TCP 203.0.113.45:49665 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:36 fw01 ACCEPT TCP 203.0.113.45:49673 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:50 fw01 ACCEPT TCP 203.0.113.45:49681 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:59 fw01 ACCEPT TCP 203.0.113.45:49689 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:10:09 fw01 ACCEPT TCP 203.0.113.45:49696 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:10:18 fw01 ACCEPT TCP 203.0.113.45:49703 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:10:32 fw01 ACCEPT TCP 203.0.113.45:49712 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:10:46 fw01 ACCEPT TCP 203.0.113.45:49713 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:01 fw01 ACCEPT TCP 203.0.113.45:49720 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:11 fw01 ACCEPT TCP 203.0.113.45:49727 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:21 fw01 ACCEPT TCP 203.0.113.45:49734 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:35 fw01 ACCEPT TCP 203.0.113.45:49742 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:48 fw01 ACCEPT TCP 203.0.113.45:49744 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:03 fw01 ACCEPT TCP 203.0.113.45:49745 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:12 fw01 ACCEPT TCP 203.0.113.45:49751 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:22 fw01 ACCEPT TCP 203.0.113.45:49753 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:36 fw01 ACCEPT TCP 203.0.113.45:49755 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:46 fw01 ACCEPT TCP 203.0.113.45:49759 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:13:00 fw01 ACCEPT TCP 203.0.113.45:49760 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:13:14 fw01 ACCEPT TCP 203.0.113.45:49766 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:13:23 fw01 ACCEPT TCP 203.0.113.45:49774 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:13:35 fw01 ACCEPT TCP 203.0.113.45:49783 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:14:06 fw01 ACCEPT TCP 203.0.113.45:50122 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:19:21 fw01 ACCEPT TCP 10.10.20.5:41766 -> 10.10.30.17:22 rule=allow-ssh-internal bytes=18844\n'
    '2026-03-13 23:40:00 fw01 DROP TCP 198.51.100.68:52943 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 23:41:00 fw01 ACCEPT UDP 10.10.30.17:53422 -> 10.10.0.2:53 rule=allow-dns-internal bytes=92\n'
    '2026-03-13 23:41:02 fw01 ACCEPT TCP 10.10.30.17:39514 -> 198.51.100.23:443 rule=allow-https-out bytes=48213904\n'
    '2026-03-13 23:50:00 fw01 DROP TCP 198.51.100.90:50796 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 23:55:00 fw01 ACCEPT TCP 10.10.50.12:45889 -> 192.0.2.19:443 rule=allow-https-out bytes=35329\n'
))

EVIDENCIA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    EVIDENCIA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


## 1. Pares pela porta de origem

O original faz `grep ':49208 '` e `grep 'port 49208 '`. Aqui fazemos para **todas** as portas de uma vez: cada porta que aparece no *firewall* e no `sshd` é uma conexão vista pelos dois relógios.


In [ ]:
import re
from collections import Counter
from datetime import datetime, timedelta, timezone

FW = EVIDENCIA["firewall.log"].splitlines()
BAS = EVIDENCIA["bastion01-auth.log"].splitlines()
FS = EVIDENCIA["fs01-auth.log"].splitlines()
ANO = 2026                                                    # o syslog não traz o ano

def fw_por_porta(linhas):
    """porta de origem -> (horário impresso, linha nº, texto)"""
    d = {}
    for n, l in enumerate(linhas, 1):
        m = re.search(r" (\d+\.\d+\.\d+\.\d+):(\d+) -> ", l)
        if m:
            d[m.group(2)] = (datetime.strptime(l[:19], "%Y-%m-%d %H:%M:%S"), n, l)
    return d

def syslog_por_porta(linhas):
    d = {}
    for n, l in enumerate(linhas, 1):
        m = re.search(r" port (\d+) ", l)
        if m:
            d.setdefault(m.group(1), (datetime.strptime(f"{ANO} {l[:15]}", "%Y %b %d %H:%M:%S"), n, l))
    return d

fw_p = fw_por_porta(FW)
bas_p = syslog_por_porta(BAS)
comuns = sorted(set(fw_p) & set(bas_p))
difs = Counter()
for porta in comuns:
    d = (bas_p[porta][0] - fw_p[porta][0]).total_seconds()
    difs[d] += 1
print(f"{len(comuns)} portas vistas pelo firewall E pelo bastion01")
print("Diferenças (bastion01 - firewall), em segundos -> quantas vezes:")
for d, n in sorted(difs.items()):
    print(f"   {d:>8.0f} s = {d / 3600:+.4f} h  ->  {n}x")

for porta in ("49208", comuns[1]):
    print(f"\nPar da porta {porta}:")
    print("  ", fw_p[porta][2][:100], f"(firewall.log:{fw_p[porta][1]})")
    print("  ", bas_p[porta][2][:100], f"(bastion01-auth.log:{bas_p[porta][1]})")


> 🔎 Todas as diferenças ficam entre 4 h + 0 s e 4 h + 1 s: um deslocamento de fuso (4 h) mais o *handshake* (≤ 1 s). Nenhum desvio de relógio entre esses dois. Agora a segunda razão independente: o que a tabela de fusos diz para Nova York em 14/mar/2026?


In [ ]:
from zoneinfo import ZoneInfo        # o zdump da biblioteca padrão

NY = ZoneInfo("America/New_York")
for dia in (datetime(2026, 3, 7, 12), datetime(2026, 3, 9, 12), datetime(2026, 3, 14, 3), datetime(2026, 11, 2, 12)):
    print(f"{dia:%Y-%m-%d}: UTC{dia.replace(tzinfo=NY).utcoffset().total_seconds() / 3600:+.0f}")
print("\nEm 14/mar/2026 Nova York está em UTC-4 (horário de verão desde 8/mar). Duas razões: dados + tabela.")


## 2. O desvio do `fs01`

O salto do `bastion01` para o `fs01` usou a porta 41766. O *firewall* (já em UTC) e o `fs01` viram a mesma conexão.


In [ ]:
fs_p = syslog_por_porta(FS)
fw_utc = fw_p["41766"][0].replace(tzinfo=NY).astimezone(timezone.utc).replace(tzinfo=None)
fs_imp = fs_p["41766"][0]
bruto = (fs_imp - fw_utc).total_seconds()
HANDSHAKE = 1
SKEW_FS01 = int(bruto - HANDSHAKE)
print(f"firewall (corrigido p/ UTC): {fw_utc:%H:%M:%S}   firewall.log:{fw_p['41766'][1]}")
print(f"fs01 imprimiu:               {fs_imp:%H:%M:%S}   fs01-auth.log:{fs_p['41766'][1]}")
print(f"diferença {bruto:.0f} s - ~{HANDSHAKE} s de handshake = fs01 adiantado ~{SKEW_FS01} s")
print("Suposição a anotar: o docportal grava no fs01, aplicamos a mesma correção SEM medir nele.")


## 3. O `merge_timeline.py` do original

Mesmo código, transformado em função que lê do dicionário `EVIDENCIA` em vez de uma pasta. As linhas comentadas mostram o que muda.


In [ ]:
import csv, io, json

def merge_timeline(ev, fs01_skew_s, fw_tz="America/New_York", ano=2026):
    FS01_SKEW = timedelta(seconds=fs01_skew_s)       # medido no passo 2
    FW_TZ = ZoneInfo(fw_tz)
    rows = []

    def syslog(nome, host, skew=timedelta(0)):
        for n, line in enumerate(ev[nome].splitlines(True), 1):
            ts = datetime.strptime(f"{ano} {line[:15]}", "%Y %b %d %H:%M:%S").replace(tzinfo=timezone.utc)
            rows.append((ts - skew, host, line[16:].strip(), f"{nome}:{n}"))

    syslog("bastion01-auth.log", "bastion01")
    syslog("fs01-auth.log", "fs01", FS01_SKEW)
    for n, line in enumerate(ev["docportal-app.jsonl"].splitlines(), 1):
        r = json.loads(line)
        ts = datetime.strptime(r["ts"], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc) - FS01_SKEW
        rows.append((ts, "fs01/docportal", f'{r["event"]} user={r["user"]} count={r.get("count")}', f"docportal-app.jsonl:{n}"))
    for n, line in enumerate(ev["firewall.log"].splitlines(), 1):
        local = datetime.strptime(line[:19], "%Y-%m-%d %H:%M:%S").replace(tzinfo=FW_TZ)
        rows.append((local.astimezone(timezone.utc), "fw01", line[20:].strip(), f"firewall.log:{n}"))

    rows.sort(key=lambda r: r[0])
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(["utc", "host", "event", "evidence_ref"])
    for ts, host, evt, ref in rows:
        w.writerow([ts.strftime("%Y-%m-%dT%H:%M:%SZ"), host, evt, ref])
    return buf.getvalue()

MERGED = merge_timeline(EVIDENCIA, SKEW_FS01)
from pathlib import Path
Path("lab-p4-merged.csv").write_text(MERGED, encoding="utf-8")
print("wc -l lab-p4-merged.csv:", len(MERGED.splitlines()), "(o original espera 297)")


## 4. Eventos materiais: bruto × corrigido

O `grep -E 'Accepted password|:41766|port 41766|docportal|tar -czf|198\.51\.100\.23' | grep -v document.view` do passo 4. Ao lado, a hora que cada fonte **imprimiu**.


In [ ]:
padrao = re.compile(r"Accepted password|:41766|port 41766|docportal|tar -czf|198\.51\.100\.23")
materiais = [l for l in MERGED.splitlines()[1:] if padrao.search(l) and "document.view" not in l]

def impresso(ref):
    """Volta ao arquivo de origem e devolve a hora como foi impressa (sem correção)."""
    nome, n = ref.rsplit(":", 1)
    linha = EVIDENCIA[nome].splitlines()[int(n) - 1]
    if nome == "docportal-app.jsonl":
        return json.loads(linha)["ts"][11:19]
    return linha[11:19] if nome == "firewall.log" else linha[7:15]

print(f"{'UTC corrigido':20} {'impresso':9} {'host':15} evento")
for l in materiais:
    r = next(csv.reader([l]))
    print(f"{r[0]:20} {impresso(r[3]):9} {r[1]:15} {r[2][:70]}  [{r[3]}]")
print("\nSem correção, o 48 MB do firewall estaria às 23:41 do dia 13: ANTES do login do atacante. Impossível.")


## 5. O grafo causal

Cada aresta tem o campo que liga. A última é tracejada: os tamanhos são próximos, mas o *firewall* não vê o conteúdo.


In [ ]:
import shutil, subprocess

DOT = """digraph lab_p4 {
  rankdir=LR; node [shape=box, fontsize=10];
  a [label="03:14:07Z bastion01\\nAccepted password svc_backup\\nfrom 203.0.113.45\\nbastion01-auth.log:117"];
  b [label="03:19:21Z fw01\\nbastion01:41766 -> fs01:22\\nfirewall.log:147"];
  c [label="03:19:22Z fs01 (skew -83s)\\nAccepted publickey svc_backup\\nfs01-auth.log:2"];
  d [label="03:23:48Z docportal (skew -83s)\\nbulk_export 412 files\\ndocportal-app.jsonl:16"];
  e [label="03:31:40Z fs01\\nsudo tar -czf /var/tmp/.q1.tgz\\nfs01-auth.log:4"];
  f [label="03:41:02Z fw01\\nfs01 -> 198.51.100.23:443\\n48,213,904 bytes\\nfirewall.log:150"];
  a -> b [label="source 10.10.20.5 = bastion01;\\nsvc_backup session open 03:14:08-03:52:44"];
  b -> c [label="src port 41766"];
  c -> d [label="user svc_backup,\\nsession open on fs01"];
  d -> e [label="path finance/2026-Q1"];
  e -> f [label="size ~ 48 MB export\\n(inferred, not matched)", style=dashed];
}"""
Path("lab-p4-timeline.dot").write_text(DOT, encoding="utf-8")

# (chr(92) + "n" é o "\n" literal dos rótulos do .dot; trocamos por espaço antes de procurar)
# Conferimos cada evidence_ref do .dot contra a linha do tempo combinada: nada de número de linha errado.
for ref in re.findall(r"([\w.-]+\.(?:log|jsonl):\d+)", DOT.replace(chr(92) + "n", " ")):
    ok = any(l.endswith(ref) for l in MERGED.splitlines())
    print(f"{'✅' if ok else '❌'} {ref}")

if shutil.which("dot"):
    subprocess.run(["dot", "-Tpng", "lab-p4-timeline.dot", "-o", "lab-p4-timeline.png"], check=False)
    print("🖼️  lab-p4-timeline.png")

NOTA = f"""# Correções de tempo (LAB-P4)

| Fonte | Correção | Par de evidências | Tamanho |
|---|---|---|---|
| firewall.log | America/New_York -> UTC | porta 49208 (firewall.log:{fw_p['49208'][1]} x bastion01-auth.log:{bas_p['49208'][1]}) e porta {comuns[1]} (firewall.log:{fw_p[comuns[1]][1]} x bastion01-auth.log:{bas_p[comuns[1]][1]}); zoneinfo confirma UTC-4 em 14/mar/2026 | +4 h |
| fs01-auth.log | relógio adiantado | porta 41766 (firewall.log:{fw_p['41766'][1]} x fs01-auth.log:{fs_p['41766'][1]}) | -{SKEW_FS01} s |
| docportal-app.jsonl | mesma do fs01 (SUPOSIÇÃO, não medida na aplicação) | - | -{SKEW_FS01} s |
| bastion01-auth.log | nenhuma (UTC); ano 2026 das notas do caso | - | 0 |

Questão em aberto (Dia 55): falhas tipo 3 contra o WS-FIN-07 vindas do bastion01 às 02:10. Nada nestes quatro logs responde.
"""
Path("correcoes-de-tempo.md").write_text(NOTA, encoding="utf-8")
print(NOTA)


## 6. ➕ Os três relógios da Serra Azul


In [ ]:
# ------------------------------------------------------------
# RECEBER FIREWALL, PROXY, WEBMAIL E SECURITY DA SERRA AZUL
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['dia-49/EVID-SA-001/fw-serraazul.log'] = ''.join((
    '2026-10-05 08:13:25 fw-sa ACCEPT TCP 10.20.40.57:57000 -> 192.0.2.25:443 rule=allow-https-out bytes=69116\n'
    '2026-10-05 08:18:47 fw-sa ACCEPT TCP 10.20.40.55:53716 -> 192.0.2.11:443 rule=allow-https-out bytes=76802\n'
    '2026-10-05 08:19:38 fw-sa ACCEPT TCP 10.20.40.57:51018 -> 198.51.100.201:80 rule=allow-http-out bytes=792\n'
    '2026-10-05 08:19:41 fw-sa ACCEPT TCP 10.20.40.57:51022 -> 203.0.113.77:443 rule=allow-https-out bytes=49960\n'
    '2026-10-05 08:21:05 fw-sa ACCEPT TCP 10.20.40.57:51031 -> 203.0.113.77:443 rule=allow-https-out bytes=7431\n'
    '2026-10-05 08:21:52 fw-sa ACCEPT TCP 10.20.40.57:51036 -> 203.0.113.77:443 rule=allow-https-out bytes=7270\n'
    '2026-10-05 08:23:12 fw-sa ACCEPT TCP 10.20.40.57:51040 -> 203.0.113.77:443 rule=allow-https-out bytes=414502\n'
    '2026-10-05 08:25:02 fw-sa ACCEPT TCP 10.20.40.57:51100 -> 198.51.100.140:443 rule=allow-https-out bytes=1994\n'
    '2026-10-05 08:26:02 fw-sa ACCEPT TCP 10.20.40.55:52944 -> 192.0.2.27:443 rule=allow-https-out bytes=75248\n'
    '2026-10-05 08:29:54 fw-sa ACCEPT TCP 10.20.40.57:51102 -> 198.51.100.140:443 rule=allow-https-out bytes=1857\n'
    '2026-10-05 08:34:49 fw-sa ACCEPT TCP 10.20.40.51:56158 -> 192.0.2.35:443 rule=allow-https-out bytes=58143\n'
    '2026-10-05 08:34:56 fw-sa ACCEPT TCP 10.20.40.57:51111 -> 198.51.100.140:443 rule=allow-https-out bytes=2284\n'
    '2026-10-05 08:39:41 fw-sa ACCEPT TCP 10.20.40.53:59046 -> 192.0.2.16:443 rule=allow-https-out bytes=7253\n'
    '2026-10-05 08:39:47 fw-sa ACCEPT TCP 10.20.40.57:51120 -> 198.51.100.140:443 rule=allow-https-out bytes=2050\n'
    '2026-10-05 08:43:55 fw-sa ACCEPT TCP 10.20.40.51:49596 -> 192.0.2.16:443 rule=allow-https-out bytes=11471\n'
    '2026-10-05 08:44:51 fw-sa ACCEPT TCP 10.20.40.57:51125 -> 198.51.100.140:443 rule=allow-https-out bytes=2025\n'
    '2026-10-05 08:48:08 fw-sa ACCEPT TCP 10.20.40.53:59292 -> 192.0.2.23:443 rule=allow-https-out bytes=62464\n'
    '2026-10-05 08:49:50 fw-sa ACCEPT TCP 10.20.40.57:51129 -> 198.51.100.140:443 rule=allow-https-out bytes=2331\n'
    '2026-10-05 08:54:59 fw-sa ACCEPT TCP 10.20.40.57:51132 -> 198.51.100.140:443 rule=allow-https-out bytes=2318\n'
    '2026-10-05 08:57:59 fw-sa ACCEPT TCP 10.20.40.52:56972 -> 192.0.2.12:443 rule=allow-https-out bytes=31257\n'
    '2026-10-05 08:59:52 fw-sa ACCEPT TCP 10.20.40.57:51139 -> 198.51.100.140:443 rule=allow-https-out bytes=2303\n'
    '2026-10-05 09:04:45 fw-sa ACCEPT TCP 10.20.40.57:51147 -> 198.51.100.140:443 rule=allow-https-out bytes=2222\n'
    '2026-10-05 09:07:10 fw-sa ACCEPT TCP 10.20.40.57:51377 -> 198.51.100.140:443 rule=allow-https-out bytes=3150210\n'
    '2026-10-05 09:09:42 fw-sa ACCEPT TCP 10.20.40.57:51154 -> 198.51.100.140:443 rule=allow-https-out bytes=1735\n'
    '2026-10-05 09:14:35 fw-sa ACCEPT TCP 10.20.40.52:49642 -> 192.0.2.22:443 rule=allow-https-out bytes=6620\n'
    '2026-10-05 09:14:49 fw-sa ACCEPT TCP 10.20.40.57:51157 -> 198.51.100.140:443 rule=allow-https-out bytes=1977\n'
    '2026-10-05 09:17:49 fw-sa ACCEPT TCP 10.20.40.52:49705 -> 192.0.2.16:443 rule=allow-https-out bytes=40378\n'
    '2026-10-05 09:19:58 fw-sa ACCEPT TCP 10.20.40.57:51161 -> 198.51.100.140:443 rule=allow-https-out bytes=2059\n'
    '2026-10-05 09:24:59 fw-sa ACCEPT TCP 10.20.40.57:51166 -> 198.51.100.140:443 rule=allow-https-out bytes=2212\n'
    '2026-10-05 09:26:36 fw-sa ACCEPT TCP 10.20.40.52:52879 -> 192.0.2.24:443 rule=allow-https-out bytes=33549\n'
    '2026-10-05 09:30:03 fw-sa ACCEPT TCP 10.20.40.57:51169 -> 198.51.100.140:443 rule=allow-https-out bytes=2222\n'
    '2026-10-05 09:30:32 fw-sa ACCEPT TCP 10.20.40.51:53871 -> 192.0.2.30:443 rule=allow-https-out bytes=8067\n'
    '2026-10-05 09:31:02 fw-sa ACCEPT TCP 203.0.113.91:52764 -> 10.20.5.20:443 rule=allow-webmail-in bytes=6120\n'
    '2026-10-05 09:31:15 fw-sa ACCEPT TCP 203.0.113.91:56244 -> 10.20.5.20:443 rule=allow-webmail-in bytes=6120\n'
    '2026-10-05 09:31:40 fw-sa ACCEPT TCP 203.0.113.91:47544 -> 10.20.5.20:443 rule=allow-webmail-in bytes=6120\n'
    '2026-10-05 09:32:05 fw-sa ACCEPT TCP 203.0.113.91:52211 -> 10.20.5.20:443 rule=allow-webmail-in bytes=88412\n'
    '2026-10-05 09:35:05 fw-sa ACCEPT TCP 10.20.40.57:51177 -> 198.51.100.140:443 rule=allow-https-out bytes=1955\n'
    '2026-10-05 09:36:01 fw-sa ACCEPT TCP 10.20.40.53:58780 -> 192.0.2.18:443 rule=allow-https-out bytes=36498\n'
))
RESERVA['dia-49/EVID-SA-001/proxy-sa.log'] = ''.join((
    'Oct  5 11:13:25 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT portal-noticias.example:443 200 out=3547 in=65569 dst=192.0.2.25\n'
    'Oct  5 11:18:47 proxy-sa squid[2211]: 10.20.40.55 atd.centro02 CONNECT portal-noticias.example:443 200 out=3123 in=73679 dst=192.0.2.11\n'
    'Oct  5 11:19:38 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 GET http://lnk.example/sa7 302 out=412 in=380 dst=198.51.100.201\n'
    'Oct  5 11:19:41 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=1840 in=48120 dst=203.0.113.77\n'
    'Oct  5 11:21:05 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=2311 in=5120 dst=203.0.113.77\n'
    'Oct  5 11:21:52 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=2290 in=4980 dst=203.0.113.77\n'
    'Oct  5 11:23:12 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=1622 in=412880 dst=203.0.113.77\n'
    'Oct  5 11:25:02 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1198 in=796 dst=198.51.100.140\n'
    'Oct  5 11:26:02 proxy-sa squid[2211]: 10.20.40.55 atd.centro02 CONNECT busca.example:443 200 out=2799 in=72449 dst=192.0.2.27\n'
    'Oct  5 11:29:54 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1227 in=630 dst=198.51.100.140\n'
    'Oct  5 11:34:49 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT portal-noticias.example:443 200 out=3094 in=55049 dst=192.0.2.35\n'
    'Oct  5 11:34:56 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1543 in=741 dst=198.51.100.140\n'
    'Oct  5 11:39:41 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT intranet-cdn.example:443 200 out=2473 in=4780 dst=192.0.2.16\n'
    'Oct  5 11:39:47 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1279 in=771 dst=198.51.100.140\n'
    'Oct  5 11:43:55 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT mapas.example:443 200 out=2981 in=8490 dst=192.0.2.16\n'
    'Oct  5 11:44:51 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1303 in=722 dst=198.51.100.140\n'
    'Oct  5 11:48:08 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT mapas.example:443 200 out=1085 in=61379 dst=192.0.2.23\n'
    'Oct  5 11:49:50 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1525 in=806 dst=198.51.100.140\n'
    'Oct  5 11:54:59 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1597 in=721 dst=198.51.100.140\n'
    'Oct  5 11:57:59 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT busca.example:443 200 out=2448 in=28809 dst=192.0.2.12\n'
    'Oct  5 11:59:52 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1477 in=826 dst=198.51.100.140\n'
    'Oct  5 12:04:45 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1323 in=899 dst=198.51.100.140\n'
    'Oct  5 12:07:10 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=3145728 in=1022 dst=198.51.100.140\n'
    'Oct  5 12:09:42 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1106 in=629 dst=198.51.100.140\n'
    'Oct  5 12:14:35 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT busca.example:443 200 out=1480 in=5140 dst=192.0.2.22\n'
    'Oct  5 12:14:49 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1277 in=700 dst=198.51.100.140\n'
    'Oct  5 12:17:49 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT intranet-cdn.example:443 200 out=2959 in=37419 dst=192.0.2.16\n'
    'Oct  5 12:19:58 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1350 in=709 dst=198.51.100.140\n'
    'Oct  5 12:24:59 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1585 in=627 dst=198.51.100.140\n'
    'Oct  5 12:26:36 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT fornecedor-cartoes.example:443 200 out=1070 in=32479 dst=192.0.2.24\n'
    'Oct  5 12:30:03 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1492 in=730 dst=198.51.100.140\n'
    'Oct  5 12:30:32 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT portal-noticias.example:443 200 out=2525 in=5542 dst=192.0.2.30\n'
    'Oct  5 12:35:05 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1101 in=854 dst=198.51.100.140\n'
    'Oct  5 12:36:01 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT mapas.example:443 200 out=2206 in=34292 dst=192.0.2.18\n'
))
RESERVA['dia-49/EVID-SA-001/webmail-sa.jsonl'] = ''.join((
    '{"ts": "2026-10-05T10:53:02Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "10.20.40.57", "result": "success"}\n'
    '{"ts": "2026-10-05T10:54:41Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-10-05T12:31:02Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:31:15Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:31:40Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:32:05Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "success"}\n'
    '{"ts": "2026-10-05T12:33:50Z", "event": "mailbox.rule_create", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "success"}\n'
    '{"ts": "2026-10-05T13:05:01Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "failure"}\n'
    '{"ts": "2026-10-05T13:05:19Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
))
RESERVA['dia-56/sa-cx-07-security.csv'] = ''.join((
    'TimeCreatedUtc,EventID,TargetUserName,LogonType,IpAddress,NewProcessName,ParentProcessName\n'
    '2026-10-05T10:51:02Z,4624,SYSTEM,5,-,"",""\n'
    '2026-10-05T10:52:10Z,4624,cx.centro07,2,-,"",""\n'
    '2026-10-05T10:52:31Z,4688,cx.centro07,,,"C:\\Windows\\explorer.exe","C:\\Windows\\System32\\userinit.exe"\n'
    '2026-10-05T10:53:40Z,4688,cx.centro07,,,"C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe","C:\\Windows\\explorer.exe"\n'
    '2026-10-05T11:24:31Z,4688,cx.centro07,,,"C:\\Users\\cx.centro07\\AppData\\Roaming\\MSA\\msa-atualizador.exe","C:\\Windows\\explorer.exe"\n'
    '2026-10-05T11:24:44Z,4688,cx.centro07,,,"C:\\Windows\\System32\\schtasks.exe","C:\\Users\\cx.centro07\\AppData\\Roaming\\MSA\\msa-atualizador.exe"\n'
    '2026-10-05T13:15:12Z,4625,cx.centro07,2,-,"",""\n'
    '2026-10-05T13:15:20Z,4624,cx.centro07,2,-,"",""\n'
    '2026-10-05T13:42:00Z,4800,cx.centro07,,,"",""\n'
))

SA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    SA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


O proxy não tem porta de origem. O par aqui é **IP de origem + IP de destino** (e, para o webmail, o IP do cliente). Vamos achar todos os pares e contar as diferenças.


In [ ]:
FW_SA = SA["fw-serraazul.log"].splitlines()
PX_SA = SA["proxy-sa.log"].splitlines()
WEB_SA = [json.loads(l) for l in SA["webmail-sa.jsonl"].splitlines() if l.strip()]

def fw_eventos(linhas):
    ev = []
    for n, l in enumerate(linhas, 1):
        m = re.search(r" (\d+\.\d+\.\d+\.\d+):\d+ -> (\d+\.\d+\.\d+\.\d+):", l)
        ev.append((datetime.strptime(l[:19], "%Y-%m-%d %H:%M:%S"), m.group(1), m.group(2), n, l))
    return ev

def px_eventos(linhas, ano):
    ev = []
    for n, l in enumerate(linhas, 1):
        src = l.split("]: ")[1].split()[0]
        dst = l.rsplit("dst=", 1)[1].strip()
        ev.append((datetime.strptime(f"{ano} {l[:15]}", "%Y %b %d %H:%M:%S"), src, dst, n, l))
    return ev

fw_ev = fw_eventos(FW_SA)
px_ev = px_eventos(PX_SA, 2026)        # o ano vem do webmail, ligado pelo par abaixo

dif = Counter()
for t_fw, s, d, *_ in fw_ev:
    for t_px, s2, d2, *_ in px_ev:
        # mesmo (origem, destino) e a diferença fica a ±2 s de um número inteiro de horas:
        # é o critério "mesmo segundo, a menos do fuso". Pares de conexões DIFERENTES ficam de fora.
        delta = (t_px - t_fw).total_seconds()
        if (s, d) == (s2, d2) and abs(delta - round(delta / 3600) * 3600) < 2:   # ±2 s de tolerância
            dif[delta] += 1
print("proxy - firewall, pares (origem, destino) no mesmo segundo a menos do fuso:")
for k, v in dif.most_common():
    print(f"   {k / 3600:+.4f} h ({k:.0f} s) -> {v} pares")

for r in WEB_SA:
    if r["src_ip"] == "203.0.113.91" and r["result"] == "success":
        t_web = datetime.strptime(r["ts"], "%Y-%m-%dT%H:%M:%SZ")
        par = [e for e in fw_ev if e[1] == "203.0.113.91" and abs((t_web - e[0]).total_seconds() - 10800) < 1]
        for e in par:
            print(f"webmail {r['ts']}  x  firewall.log:{e[3]} {e[0]:%H:%M:%S} -> diferença {(t_web - e[0]).total_seconds() / 3600:+.4f} h")

SP = ZoneInfo("America/Sao_Paulo")
offset_sp = datetime(2026, 10, 5, 12, tzinfo=SP).utcoffset().total_seconds() / 3600
print(f"\nzoneinfo: São Paulo em 5/out/2026 = UTC{offset_sp:+.0f} (sem horário de verão desde 2019)")
print("Duas fontes de pares + tabela de fusos: firewall = America/Sao_Paulo, desvio de relógio 0 s.")


In [ ]:
def merge_sa(sa):
    rows = []
    for t, s, d, n, l in fw_eventos(sa["fw-serraazul.log"].splitlines()):
        rows.append((t.replace(tzinfo=SP).astimezone(timezone.utc), "fw-sa", l[20:].strip(), f"fw-serraazul.log:{n}"))
    for t, s, d, n, l in px_eventos(sa["proxy-sa.log"].splitlines(), 2026):
        rows.append((t.replace(tzinfo=timezone.utc), "proxy-sa", l.split("]: ")[1].strip(), f"proxy-sa.log:{n}"))
    for n, r in enumerate((json.loads(x) for x in sa["webmail-sa.jsonl"].splitlines() if x.strip()), 1):
        t = datetime.strptime(r["ts"], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
        rows.append((t, "webmail-sa", f"{r['event']} user={r['user']} src={r['src_ip']} {r['result']}", f"webmail-sa.jsonl:{n}"))
    for n, r in enumerate(csv.DictReader(io.StringIO(sa["sa-cx-07-security.csv"])), 2):
        t = datetime.strptime(r["TimeCreatedUtc"], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
        proc = r["NewProcessName"].split("\\")[-1]
        rows.append((t, "SA-CX-07", f"{r['EventID']} {r['TargetUserName']} {proc}".strip(), f"sa-cx-07-security.csv:{n}"))
    rows.sort(key=lambda r: (r[0], r[1]))
    return rows

SA_ROWS = merge_sa(SA)
buf = io.StringIO(); w = csv.writer(buf); w.writerow(["utc", "host", "event", "evidence_ref"])
for t, h, e, ref in SA_ROWS:
    w.writerow([t.strftime("%Y-%m-%dT%H:%M:%SZ"), h, e, ref])
Path("serraazul-merged.csv").write_text(buf.getvalue(), encoding="utf-8")

MAT = re.compile(r"lnk\.example|203\.0\.113\.77|serraazul-pix|msa-atualizador|schtasks|out=3145728|bytes=3150210|203\.0\.113\.91|rule_create")
print(f"{len(SA_ROWS)} linhas. Eventos materiais (UTC):\n")
primeiro_beacon = True
for t, h, e, ref in SA_ROWS:
    if MAT.search(e) or (primeiro_beacon and h == "proxy-sa" and "atualiza-modulo" in e):
        if "atualiza-modulo" in e and "out=3145728" not in e:
            primeiro_beacon = False          # só o primeiro da série; os outros você viu no Dia 54
        print(f"{t:%H:%M:%S}  {h:11} {e[:80]:80} [{ref}]")


> ✅ Na ordem certa: o clique (11:19:38) vem **antes** do executável (11:24:31), que vem antes do primeiro *beacon* (11:25:02), do envio de 3 MiB (12:07:10) e do *login* externo (12:32:05). A planilha da abertura estava errada por exatamente 3 h.

### O grafo do caso BR, com as arestas honestas


In [ ]:
DOT_SA = """digraph serra_azul {
  rankdir=LR; node [shape=box, fontsize=10];
  a [label="11:19:38Z proxy/fw\\nGET lnk.example/sa7 (302)\\nsrc 10.20.40.57"];
  b [label="11:19:41-11:23:12Z proxy/fw\\nCONNECT serraazul-pix.example\\n203.0.113.77"];
  c [label="11:24:31Z SA-CX-07 4688\\nmsa-atualizador.exe\\npai explorer.exe"];
  d [label="11:24:44Z SA-CX-07 4688\\nschtasks.exe\\npai msa-atualizador.exe"];
  e [label="11:25:02Z ... a cada ~300 s\\nCONNECT atualiza-modulo.example\\n198.51.100.140"];
  f [label="12:07:10Z proxy/fw\\n3.145.728 bytes -> 198.51.100.140"];
  g [label="12:32:05Z webmail/fw\\nlogin cx.centro07\\nde 203.0.113.91"];
  h [label="12:33:50Z webmail\\nmailbox.rule_create"];
  a -> b [label="redirect 302; src 10.20.40.57"];
  b -> c [label="próximo no tempo; ZIP na MFT (Dia 52),\\nproxy não mostra o nome do arquivo", style=dashed];
  c -> d [label="PID pai (4688 e psscan)"];
  c -> e [label="nome atualiza-modulo.example\\n(--canal, Dia 53) = destino do proxy"];
  e -> f [label="dst 198.51.100.140; conteúdo NÃO observado"];
  b -> g [label="senha digitada no site falso não aparece\\nem log nenhum: inferido pela conta e pela ordem", style=dashed];
  g -> h [label="mesma sessão: user + src 203.0.113.91"];
}"""
Path("serraazul-timeline.dot").write_text(DOT_SA, encoding="utf-8")
if shutil.which("dot"):
    subprocess.run(["dot", "-Tpng", "serraazul-timeline.dot", "-o", "serraazul-timeline.png"], check=False)
    print("🖼️  serraazul-timeline.png")
print("Arestas tracejadas:", DOT_SA.count("style=dashed"), "(cada uma com o motivo no rótulo)")


## 7. 🎮 Quiz do relógio


In [ ]:
MODO_JOGO = True
QUIZ = [
    ("LAB-P4: alguém supõe o firewall em UTC-5 (não UTC-4). Os 48 MB (23:41:02 impresso) vão para...",
     ["03:41:02 UTC", "04:41:02 UTC", "02:41:02 UTC"], 2,
     "UTC-5 soma 5 h: 04:41:02, uma hora depois do real. É o evento que mais 'anda' na história."),
    ("➕ O firewall da Serra Azul lido como UTC: o login externo (09:32:05 impresso) fica...",
     ["antes do clique das 11:19", "depois do clique", "no mesmo minuto"], 1,
     "É o erro da planilha da abertura."),
    ("Duas linhas a 2 s uma da outra em hosts diferentes, sem campo em comum. No grafo elas...",
     ["ganham aresta sólida", "ficam sem aresta, ou tracejada com ?", "viram um nó só"], 2,
     "Adjacente não é ligado."),
    ("O docportal recebe a correção de -83 s do fs01. Isso é...",
     ["medido", "uma suposição a anotar", "desnecessário"], 2, "Mesmo host, mas não medido na aplicação."),
    ("➕ De onde vem o ano do proxy da Serra Azul?",
     ["do palpite", "do webmail, ligado ao firewall/proxy por pares", "do nome do arquivo"], 2,
     "Fonte com data completa, ligada por um par de eventos."),
]
acertos = 0
for i, (p, ops, certa, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}] {p}")
    for k, o in enumerate(ops, 1):
        print(f"   {k}. {o}")
    if MODO_JOGO:
        ok = input("   Resposta: ").strip() == str(certa)
        acertos += ok
        print("   ✅" if ok else "   ❌", end=" ")
    print(f"Certa: {certa}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")


In [ ]:
try:
    from google.colab import files
    for f in ("lab-p4-merged.csv", "lab-p4-timeline.dot", "correcoes-de-tempo.md", "serraazul-merged.csv", "serraazul-timeline.dot"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: arquivos na pasta atual)")


## ✅ Antes de fechar o Dia 56

- [ ] nota de correções com dois pares de portas para UTC-4 e o par 41766 para o `fs01`
- [ ] `lab-p4-merged.csv` com 297 linhas e `evidence_ref`
- [ ] grafo com todas as arestas rotuladas e uma inferida, justificada
- [ ] questão em aberto do Dia 55 ainda aberta
- [ ] ➕ Serra Azul: dois pares (fw × proxy, fw × webmail), ano do proxy justificado, três arestas tracejadas

**Amanhã (Dia 57):** super-linhas do tempo com o Plaso, e a pergunta que Rafa vai fazer: *"quantos eventos do firewall tem aí dentro?"*

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>
